In [6]:
# process_billion.py
import os
import csv
import re
import time
import traceback
import random
import sys

# Try import and give helpful message if it fails
try:
    from generating_synthetic_noise import SpanishNoiseSynthesizer, Config
except Exception as e:
    print("ERROR: couldn't import 'generating_synthetic_noise'. Make sure it's on PYTHONPATH.")
    print("Import error:", e)
    print("sys.path:", sys.path)
    raise

def merge_hyphenated_words(text):
    # Remove hyphens at line breaks (e.g., "examp-\nle" -> "example")
    return re.sub(r'-\s*\n\s*', '', text)

def split_sentences(text):
    # Split on sentence-ending punctuation (., !, ?) followed by whitespace/newline.
    # Keeps the punctuation at the end of each sentence.
    parts = re.split(r'(?<=[.!?])\s+', text)
    # Also collapse empty and whitespace-only entries
    sentences = [p.strip() for p in parts if p and p.strip()]
    return sentences

def process_file(input_path, output_path, seed=42, debug_max_sentences=None):
    start = time.time()
    print(f"\n--- Processing file: {input_path}")
    if not os.path.exists(input_path):
        print("Input file does not exist:", input_path)
        return 0

    # Read file (robust to encoding issues)
    with open(input_path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()
    print(f"Read {len(text)} chars from file")

    if not text.strip():
        print("File is empty or whitespace-only; skipping.")
        return 0

    text = merge_hyphenated_words(text)
    sentences = split_sentences(text)
    print(f"Split into {len(sentences)} sentence candidates")

    # init synthesizer (try with seed, else fallback)
    try:
        synthesizer = SpanishNoiseSynthesizer(seed=seed)
    except Exception as e:
        print("Warning: SpanishNoiseSynthesizer(seed=...) failed, trying without seed. Error:", e)
        try:
            synthesizer = SpanishNoiseSynthesizer()
        except Exception as e2:
            print("ERROR: Could not construct SpanishNoiseSynthesizer:", e2)
            raise

    # choose RNG: synthesizer.rng if available else python random.Random
    if hasattr(synthesizer, "rng") and getattr(synthesizer, "rng") is not None:
        rng = synthesizer.rng
        print("Using synthesizer.rng for randomness")
    else:
        rng = random.Random(seed)
        print("Using local random.Random for randomness (synth has no .rng)")

    # build enabled_rules robustly
    try:
        enabled_rules = {rule: True for rule in Config.RULE_PROBABILITIES if rule != "identity"}
    except Exception:
        # fallback if Config doesn't have RULE_PROBABILITIES attribute
        try:
            enabled_rules = {k: True for k in getattr(Config, "RULES", [])}
            print("Warning: Config.RULE_PROBABILITIES not found, used Config.RULES fallback")
        except Exception:
            enabled_rules = {}
            print("Warning: Could not build enabled_rules from Config. Proceeding with empty rules dict.")

    pairs = []
    for i, sent in enumerate(sentences):
        if debug_max_sentences and i >= debug_max_sentences:
            break
        clean_sent = sent.strip()
        if len(clean_sent) < 3:
            continue

        # choose random number of edits; use rng.choice if available, else fallback
        try:
            num_edits = rng.choice([5,6,7,8,9,10])
        except Exception:
            # rng might be random.Random or something else; fallback to Python random
            num_edits = random.choice([5,6,7,8,9,10])

        noisy = clean_sent
        try:
            for _ in range(num_edits):
                res = synthesizer.generate_noisy_variant(noisy, enabled_rules)
                # res may be (noisy_text, rules) or a single string
                if isinstance(res, (tuple, list)) and len(res) >= 1:
                    noisy = res[0]
                elif isinstance(res, str):
                    noisy = res
                else:
                    # unexpected; try converting to str
                    noisy = str(res)
        except Exception as e:
            print(f"generate_noisy_variant raised on sentence idx {i}: {e}")
            traceback.print_exc()
            continue

        noisy = noisy.strip()
        if noisy and noisy != clean_sent:
            pairs.append((noisy, clean_sent))

    # write TSV if we have pairs
    if pairs:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f, delimiter="\t")
            writer.writerow(["Input", "Target"])
            for noisy, target in pairs:
                writer.writerow([noisy, target])
        print(f"Wrote {len(pairs)} pairs to {output_path}")
    else:
        print("No noisy pairs generated for this file.")

    end = time.time()
    print(f"File done in {end - start:.2f}s")
    return len(pairs)

def main():
    input_dir = r"C:\Users\prana\Downloads\clean_corpus\spanish_billion_words"
    output_dir = r"C:\Users\prana\Downloads\clean_corpus\processed_billion_words"
    DEBUG_MAX_SENTENCES = None   # set to e.g. 100 for quicker test

    if not os.path.isdir(input_dir):
        print("ERROR: input_dir not found:", input_dir)
        return

    os.makedirs(output_dir, exist_ok=True)

    # grab ALL files, not just .txt
    files = [f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir, f))]
    print(f"Found {len(files)} files in: {input_dir}")

    if not files:
        print("No files found. Directory listing:")
        for e in os.listdir(input_dir):
            print(" ", e)
        return

    total_pairs = 0
    for fn in files:
        inpath = os.path.join(input_dir, fn)
        outfn = os.path.splitext(fn)[0] + ".tsv"  # always save with .tsv
        outpath = os.path.join(output_dir, outfn)

        print(f"Processing {fn}...")
        pairs = process_file(inpath, outpath, seed=42, debug_max_sentences=DEBUG_MAX_SENTENCES)
        total_pairs += pairs

    print(f"\n=== DONE ===\nProcessed {len(files)} files, total pairs: {total_pairs}")


In [7]:

if __name__ == "__main__":
    main()


Found 100 files in: C:\Users\prana\Downloads\clean_corpus\spanish_billion_words
Processing spanish_billion_words_00...

--- Processing file: C:\Users\prana\Downloads\clean_corpus\spanish_billion_words\spanish_billion_words_00
Read 69193517 chars from file
Split into 1 sentence candidates
Using synthesizer.rng for randomness
Wrote 1 pairs to C:\Users\prana\Downloads\clean_corpus\processed_billion_words\spanish_billion_words_00.tsv
File done in 30.36s
Processing spanish_billion_words_01...

--- Processing file: C:\Users\prana\Downloads\clean_corpus\spanish_billion_words\spanish_billion_words_01
Read 70911626 chars from file
Split into 1 sentence candidates
Using synthesizer.rng for randomness
Wrote 1 pairs to C:\Users\prana\Downloads\clean_corpus\processed_billion_words\spanish_billion_words_01.tsv
File done in 65.35s
Processing spanish_billion_words_02...

--- Processing file: C:\Users\prana\Downloads\clean_corpus\spanish_billion_words\spanish_billion_words_02
Read 73195755 chars from f